In [1]:
import os
import spacy
import re
import contractions
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI

d:\Study\pyspider\GenAI\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\HP\AppData\Local\Temp\ipykernel_19012\2046611164.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [2]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [3]:
with open('data.txt', 'r') as file:
    data = file.read().lower()

In [4]:
data = contractions.fix(data)

In [5]:
data = re.sub('[^0-9a-zA-Z\s]', "", data).strip()

In [6]:
data = re.sub('[0-9]', "", data)

In [7]:
data = re.sub('\s+', " ", data)
data

'artificial intelligence ai machine learning ml and deep learning dl a complete guide table of contents introduction what is artificial intelligence ai history and evolution of ai types of artificial intelligence what is machine learning ml how machine learning works types of machine learning supervised learning unsupervised learning semisupervised learning reinforcement learning common machine learning algorithms what is deep learning dl how deep learning works neural network architectures difference between ai ml and dl applications of ai ml and dl popular tools and frameworks challenges and limitations ethics and responsible ai future trends conclusion glossary of terms introduction artificial intelligence machine learning and deep learning are three of the most talkedabout technologies of the twentyfirst century they are reshaping industries changing how businesses operate and influencing everyday life in ways most people do not even notice from voice assistants like siri and alexa

In [8]:
nlp = spacy.load('en_core_web_sm')

In [9]:
tokens = nlp(data)
lemmatize_tokens = [token.lemma_ for token in tokens if not token.is_stop]
data = " ".join(lemmatize_tokens).strip()
data

'artificial intelligence ai machine learn ml deep learning dl complete guide table content introduction artificial intelligence ai history evolution ai type artificial intelligence machine learn ml machine learn work type machine learn supervise learning unsupervised learning semisupervise learn reinforcement learn common machine learning algorithm deep learning dl deep learning work neural network architecture difference ai ml dl application ai ml dl popular tool framework challenge limitation ethic responsible ai future trend conclusion glossary term introduction artificial intelligence machine learning deep learning talkedabout technology twentyfirst century reshape industry change business operate influence everyday life way people notice voice assistant like siri alexa recommendation system netflix amazon selfdrive car medical diagnosis tool technology term interchangeably casual conversation thing artificial intelligence broad concept machine learning subset ai deep learning subs

In [10]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200, 
    chunk_overlap=40, 
    )
data = text_splitter.create_documents([data])
data

[Document(metadata={}, page_content='artificial intelligence ai machine learn ml deep learning dl complete guide table content introduction artificial intelligence ai history evolution ai type artificial intelligence machine learn ml'),
 Document(metadata={}, page_content='intelligence machine learn ml machine learn work type machine learn supervise learning unsupervised learning semisupervise learn reinforcement learn common machine learning algorithm deep learning dl'),
 Document(metadata={}, page_content='learning algorithm deep learning dl deep learning work neural network architecture difference ai ml dl application ai ml dl popular tool framework challenge limitation ethic responsible ai future'),
 Document(metadata={}, page_content='limitation ethic responsible ai future trend conclusion glossary term introduction artificial intelligence machine learning deep learning talkedabout technology twentyfirst century reshape industry'),
 Document(metadata={}, page_content='twentyfirst 

In [11]:
embeddings_model = HuggingFaceEmbeddings(
    model_name = 'sentence-transformers/all-miniLM-L6-V2'
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1750.10it/s]


In [12]:
vectordb = FAISS.from_documents(documents=data, embedding=embeddings_model)
vectordb

In [67]:
verification_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a document verifier."
    "Your task is to verify whether the retrieved document is relevant to the user's question."
    "check the retrieved chunk score in percentage,"

    "example, question - what is machine learning, answer - Deep learning is the subset of ai."
    "wrong answer"
    "retrived out of 5 chunks 4 chunks are correct. return the score: "
    "if any chunk is wrongle retrieved, detect and mention which chunk is the mistake, and explain why it is wrong"

    "don't generate the output only returns the doc which is related to the user question"),
    
    ("human","User question: {query}"
      "Retrieved document:{r_chunks} "
    )
])

verification_model = ChatGoogleGenerativeAI(
    model = 'gemini-3.1-flash-lite',
    api_key = os.environ['GEMINI_API_KEY'],
    temperature = 0.0
    )

parser = StrOutputParser()

In [68]:
verification_chain = verification_prompt | verification_model | parser

In [69]:
def rag_query(query, k=5):
    r_chunks = vectordb.similarity_search(query)
    r_chunks = [doc.page_content for doc in r_chunks]
    r_string = " ".join(r_chunks)

    for idx, doc in enumerate(r_chunks):
        print(idx, doc)

    response = verification_chain.invoke({'query' : query,'r_chunks':r_chunks})

    return response
user_prompt = "Explain GenAi ?"
user_prompt = re.sub('[^0-9a-zA-Z\s]', "", user_prompt)

response = rag_query(user_prompt)
print(response)

0 complex multistep task minimal human intervention explainable ai xai development technique ai decisionmake transparent interpretable ai scientific discovery ai accelerate research field like drug
1 ai general ai refer theoretical form ai possess ability understand learn apply knowledge wide range task similar human intelligence form ai exist c super ai super ai hypothetical form ai surpass
2 language model llm generative ai tool capable produce humanlike text image audio video mark new chapter ai history bring ai mainstream use million people worldwide type artificial intelligence ai
3 large language model llm like gpt claude generative adversarial network gan consist neural network generator discriminator compete produce realistic synthetic datum image autoencoder neural network
**Score: 75%**

**Analysis:**
*   **Chunk 1:** Irrelevant. It discusses XAI and scientific discovery, which does not explain what Generative AI is.
*   **Chunk 2:** Relevant. It defines AI and mentions the s